In [1]:
import requests
import xmltodict

from collections import Counter
from dataclasses import dataclass, field
from typing import Literal

import numpy as np
import arff
from tempfile import NamedTemporaryFile

# Helpers

In [2]:
def download_and_parse(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    return xmltodict.parse(response.content)

def download_to_temp_file(
    url: str,
    suffix: str = "",
    chunk_size: int = 8192,
) -> str:
    """
    Download a URL to a temporary file.

    Returns
    -------
    str
        Path to the downloaded file.
    """
    with requests.get(url, stream=True) as response:
        response.raise_for_status()

        with NamedTemporaryFile(
            suffix=suffix,
            delete=False,
        ) as tmp:
            for chunk in response.iter_content(chunk_size=chunk_size):
                tmp.write(chunk)

    return tmp.name

In [44]:
# ============================================================================
# Dataset metadata
# ============================================================================

@dataclass(slots=True, frozen=True)
class DatasetDownloadInfo:
    file_path: str
    default_target_attribute: str | None


# ============================================================================
# Feature models
# ============================================================================
@dataclass(slots = True)
class Quality:
    name: str
    value: float |None = None

    def __str__(self) -> str:
        return f"{self.name} - {self.value}"




@dataclass(slots=True)
class Feature:
    index: int
    name: str
    data_type: str

    nominal_values: list[str] = field(default_factory=list)

    is_target: bool = False
    is_ignore: bool = False
    is_row_identifier: bool = False

    number_of_distinct_values: int | None = None
    number_of_unique_values: int | None = None
    number_of_missing_values: int | None = None
    number_of_integer_values: int | None = None
    number_of_real_values: int | None = None
    number_of_nominal_values: int | None = None
    number_of_values: int | None = None

    maximum_value: float | None = None
    minimum_value: float | None = None
    mean_value: float | None = None
    standard_deviation: float | None = None

    class_distribution: str | None = None

    def __str__(self) -> str:
        return f"{self.index} - {self.name}"


@dataclass(slots=True)
class DataFeature:
    did: int | None = None
    evaluation_engine_id: int | None = None
    features: list[Feature] = field(default_factory=list)
    qualities: list[Quality] = field(default_factory=list)
    error: str | None = None

    def feature_map(
        self,
        *,
        sorted_names: bool = False,
    ) -> dict[str, Feature]:
        result = {f.name: f for f in self.features}
        return dict(sorted(result.items())) if sorted_names else result


# ============================================================================
# Constants
# ============================================================================

_NUMERIC_TYPES = frozenset({"NUMERIC", "REAL", "INTEGER"})


_OML_BOOL_FIELDS = (
    "is_target",
    "is_ignore",
    "is_row_identifier",
)

_OML_INT_FIELDS = (
    "number_of_missing_values",
    "number_of_distinct_values",
    "number_of_unique_values",
    "number_of_integer_values",
    "number_of_real_values",
    "number_of_nominal_values",
    "number_of_values",
)

_OML_FLOAT_FIELDS = (
    "maximum_value",
    "minimum_value",
    "mean_value",
    "standard_deviation",
)


# ============================================================================
# Dataset retrieval 
# ============================================================================

def get_data_and_meta_information_from_did(
    did: int,
    dataset_type: Literal["arff", "parquet"] = "arff",
) -> DatasetDownloadInfo:
    dataset_type = dataset_type.lower()

    if dataset_type not in {"arff", "parquet"}:
        raise ValueError("dataset_type must be 'arff' or 'parquet'")

    metadata = download_and_parse(
        f"https://www.openml.org/api/v1/xml/data/{did}"
    )["oml:data_set_description"]

    url_key = "oml:url" if dataset_type == "arff" else "oml:parquet_url"

    return DatasetDownloadInfo(
        file_path=download_to_temp_file(
            metadata[url_key],
            suffix=f".{dataset_type}",
        ),
        default_target_attribute=metadata.get(
            "oml:default_target_attribute"
        ),
    )


# ============================================================================
# ARFF type handling
# ============================================================================

def _liac_type(type_spec):
    if isinstance(type_spec, list):
        return "nominal", tuple(type_spec)

    normalized = type_spec.upper()

    if normalized in _NUMERIC_TYPES:
        return "numeric", None

    return {
        "STRING": ("string", None),
        "DATE": ("date", None),
    }.get(normalized, (normalized.lower(), None))


def _normalize_target_names(target: str | list[str] | None) -> set[str]:
    if target is None:
        return set()

    if isinstance(target, str):
        return {t.strip() for t in target.split(",") if t.strip()}

    return {t.strip() for t in target if t and t.strip()}


# ============================================================================
# Feature extraction
# ============================================================================

def _fill_numeric(col, feat: Feature) -> None:
    feat.number_of_values = len(col)

    present = np.asarray(
        [v for v in col if v is not None],
        dtype=np.float64,
    )

    feat.number_of_missing_values = len(col) - present.size

    if present.size == 0:
        return

    int_mask = present == np.floor(present)

    feat.number_of_integer_values = int(int_mask.sum())
    feat.number_of_real_values = int((~int_mask).sum())

    feat.maximum_value = float(np.max(present))
    feat.minimum_value = float(np.min(present))
    feat.mean_value = float(np.mean(present))
    feat.standard_deviation = float(np.std(present, ddof=0))

    unique_vals, counts = np.unique(
        present,
        return_counts=True,
    )

    feat.number_of_distinct_values = len(unique_vals)
    feat.number_of_unique_values = int(np.sum(counts == 1))


def _fill_nominal(col, feat: Feature, schema_values) -> None:
    feat.nominal_values = sorted(schema_values)
    feat.number_of_values = len(col)

    counts = Counter(
        v for v in col if v is not None
    )

    present = sum(counts.values())

    feat.number_of_missing_values = len(col) - present
    feat.number_of_nominal_values = present
    feat.number_of_distinct_values = len(counts)
    feat.number_of_unique_values = sum(c == 1 for c in counts.values())

    feat.class_distribution = ",".join(
        f"{k}:{v}" for k, v in sorted(counts.items())
    )


# ============================================================================
# MAIN LOADER 
# ============================================================================

def load_arff_features(
    dataset: DatasetDownloadInfo,
    *,
    did: int | None = None,
    evaluation_engine_id: int | None = None,
) -> DataFeature:
    try:
        with open(
            dataset.file_path,
            "r",
            encoding="utf-8",
            errors="replace",
        ) as f:
            data = arff.load(f)

    except Exception as exc:
        return DataFeature(
            did=did,
            evaluation_engine_id=evaluation_engine_id,
            error=str(exc),
        )

    attributes = data["attributes"]
    rows = data["data"]

    target_names = _normalize_target_names(
        dataset.default_target_attribute
    )

    if rows:
        columns = list(zip(*rows))
    else:
        columns = [tuple() for _ in attributes]

    features: list[Feature] = []

    for idx, ((name, type_spec), col) in enumerate(
        zip(attributes, columns)
    ):
        type_name, type_range = _liac_type(type_spec)

        feat = Feature(
            index=idx,
            name=name,
            data_type=type_name,
            is_target=name in target_names,
        )

        if type_name == "numeric":
            _fill_numeric(col, feat)

        elif type_name == "nominal":
            _fill_nominal(col, feat, type_range)

        else:
            feat.number_of_values = len(col)

        features.append(feat)

    return DataFeature(
        did=did,
        evaluation_engine_id=evaluation_engine_id,
        features=features,
    )


# ============================================================================
# SERIALIZATION 
# ============================================================================

def quality_to_oml_dict(qua: Quality) -> dict:
    result = {
        "oml:name": qua.name,
        "oml:value": qua.value,
    }

    for attr in (*_OML_INT_FIELDS, *_OML_FLOAT_FIELDS):
        v = getattr(qua, attr)
        if v is not None:
            result[f"oml:{attr}"] = str(v)

    return result

def qualities_to_oml_dict(data_feature: DataFeature) -> dict:
    return {
        "oml:data_qualities": {
            "@xmlns:oml": "http://openml.org/openml",
            "oml:quality": [
                quality_to_oml_dict(f)
                for f in data_feature.qualities
            ],
        }
    }


def features_to_xml(
    data_feature: DataFeature,
    *,
    pretty: bool = True,
) -> str:
    return xmltodict.unparse(
        qualities_to_oml_dict(data_feature),
        pretty=pretty,
    )


def parse_features_xml(xml_str: str) -> dict:
    return xmltodict.parse(
        xml_str,
        force_list=("oml:feature", "oml:nominal_value"),
    )

In [45]:
from pymfe.mfe import MFE
import pandas as pd

In [46]:
data_id = 200

In [47]:
qualities_url = lambda data_id:f"https://www.openml.org/api/v1/data/qualities/{data_id}"
server_features_qualtiies = lambda data_id:download_and_parse(qualities_url(data_id))

In [48]:
dataset = get_data_and_meta_information_from_did(data_id)

In [49]:
dataset

DatasetDownloadInfo(file_path='/var/folders/_f/ng_zp8zj2dgf828sb6s5wdb00000gn/T/tmpg13lj0lw.arff', default_target_attribute='class')

In [50]:
qualities_arff = server_features_qualtiies(data_id=data_id)
qualities_arff

{'oml:data_qualities': {'@xmlns:oml': 'http://openml.org/openml',
  'oml:quality': [{'oml:name': 'AutoCorrelation',
    'oml:value': '-1003.4172661870504'},
   {'oml:name': 'CfsSubsetEval_DecisionStumpAUC', 'oml:value': None},
   {'oml:name': 'CfsSubsetEval_DecisionStumpErrRate', 'oml:value': None},
   {'oml:name': 'CfsSubsetEval_DecisionStumpKappa', 'oml:value': None},
   {'oml:name': 'CfsSubsetEval_NaiveBayesAUC', 'oml:value': None},
   {'oml:name': 'CfsSubsetEval_NaiveBayesErrRate', 'oml:value': None},
   {'oml:name': 'CfsSubsetEval_NaiveBayesKappa', 'oml:value': None},
   {'oml:name': 'CfsSubsetEval_kNN1NAUC', 'oml:value': None},
   {'oml:name': 'CfsSubsetEval_kNN1NErrRate', 'oml:value': None},
   {'oml:name': 'CfsSubsetEval_kNN1NKappa', 'oml:value': None},
   {'oml:name': 'ClassEntropy', 'oml:value': None},
   {'oml:name': 'DecisionStumpAUC', 'oml:value': None},
   {'oml:name': 'DecisionStumpErrRate', 'oml:value': None},
   {'oml:name': 'DecisionStumpKappa', 'oml:value': None},
  

In [51]:

with open(
    dataset.file_path,
    "r",
    encoding="utf-8",
    errors="replace",
) as f:
    data = arff.load(f)


In [52]:

attributes = [attr[0] for attr in data['attributes']]
target_idx = attributes.index(dataset.default_target_attribute)
arr = np.array(data['data'])

col_names = [attr for i, attr in enumerate(attributes) if i != target_idx]
X_full = pd.DataFrame(np.delete(arr, target_idx, axis=1), columns=col_names)
y_raw = arr[:, target_idx]

if np.issubdtype(np.array(y_raw, dtype=object).dtype, np.str_) or np.array(y_raw, dtype=object).dtype == object:
    y = pd.Categorical(y_raw).codes.astype(float)
else:
    y = y_raw.astype(float)

string_cols = X_full.select_dtypes(include='str').columns
multi_word_cols = [col for col in string_cols if X_full[col].dropna().str.contains(r'\s').any()]
X = X_full.drop(columns=multi_word_cols).to_numpy()
X_clean = X_full.drop(columns=multi_word_cols)

# Encode remaining object/string columns
for col in X_clean.select_dtypes(include=['object', 'string']).columns:
    X_clean[col] = pd.Categorical(X_clean[col]).codes.astype(float)

X = X_clean.to_numpy(dtype=float)

In [53]:
X

array([[  1.,   0., 267., ...,  66.,  32.,   3.],
       [  0.,   0., 249., ...,  91.,  16.,   2.],
       [  1.,   0., 334., ...,  42.,  30.,   3.],
       ...,
       [  0.,  -1., 256., ...,  36.,   9.,  -1.],
       [  0.,  -1., 260., ..., 126.,  14.,  -1.],
       [  0.,  -1., 207., ..., 189.,  16.,  -1.]], shape=(418, 18))

In [54]:
all_groups = ["general", "statistical", "info-theory", "model-based", "landmarking", "relative", "clustering", "concept", "itemset", "complexity"]
important_groups = ["general", "statistical", "info-theory"]

In [55]:
mfe = MFE(
    groups=important_groups,
    random_state=42, 
)
mfe.fit(X, y)
ft = mfe.extract(cat_cols='auto', suppress_warnings=True, verbose = 1, timeout=30)
print("\n".join("{:50} {:30}".format(x, y) for x, y in zip(ft[0], ft[1])))

/Users/smukherjee/Documents/Projects/OpenML/EvaluationEnginePython/.venv/lib/python3.13/site-packages/pymfe/_internal.py:1567: UserWarning: It is not possible make equal discretization
  warnings.warn("It is not possible make equal discretization")


  0%|          | 0/48 [00:00<?, ?it/s]

Process of metafeature extraction finished.                                     
attr_conc.mean                                                0.14259062745639062
attr_conc.sd                                                   0.1822484717004583
attr_ent.mean                                                  2.0950083256633167
attr_ent.sd                                                    0.7512220510971301
attr_to_inst                                                   0.0430622009569378
can_cor.mean                                                    0.977555639526626
can_cor.sd                                                   0.023823506572091025
cat_to_num                                                                    0.0
class_conc.mean                                                0.9532227342393266
class_conc.sd                                                0.007578272804411549
class_ent                                                       8.616450041171792
cor.mean         

In [ ]:
from pymfe.mfe import MFE
import pandas as pd

data_id = 200

qualities_url = lambda data_id:f"https://www.openml.org/api/v1/data/qualities/{data_id}"
server_features_qualtiies = lambda data_id:download_and_parse(qualities_url(data_id))

dataset = get_data_and_meta_information_from_did(data_id)

dataset

qualities_arff = server_features_qualtiies(data_id=data_id)
qualities_arff


with open(
    dataset.file_path,
    "r",
    encoding="utf-8",
    errors="replace",
) as f:
    data = arff.load(f)



attributes = [attr[0] for attr in data['attributes']]
target_idx = attributes.index(dataset.default_target_attribute)
arr = np.array(data['data'])

col_names = [attr for i, attr in enumerate(attributes) if i != target_idx]
X_full = pd.DataFrame(np.delete(arr, target_idx, axis=1), columns=col_names)
y_raw = arr[:, target_idx]

if np.issubdtype(np.array(y_raw, dtype=object).dtype, np.str_) or np.array(y_raw, dtype=object).dtype == object:
    y = pd.Categorical(y_raw).codes.astype(float)
else:
    y = y_raw.astype(float)

string_cols = X_full.select_dtypes(include='str').columns
multi_word_cols = [col for col in string_cols if X_full[col].dropna().str.contains(r'\s').any()]
X = X_full.drop(columns=multi_word_cols).to_numpy()
X_clean = X_full.drop(columns=multi_word_cols)

# Encode remaining object/string columns
for col in X_clean.select_dtypes(include=['object', 'string']).columns:
    X_clean[col] = pd.Categorical(X_clean[col]).codes.astype(float)

X = X_clean.to_numpy(dtype=float)

X

all_groups = ["general", "statistical", "info-theory", "model-based", "landmarking", "relative", "clustering", "concept", "itemset", "complexity"]
important_groups = ["general", "statistical", "info-theory"]

mfe = MFE(
    groups=important_groups,
    random_state=42, 
)
mfe.fit(X, y)
ft = mfe.extract(cat_cols='auto', suppress_warnings=True, verbose = 1, timeout=30)
print("\n".join("{:50} {:30}".format(x, y) for x, y in zip(ft[0], ft[1])))

In [ ]:
data_id = 200
arff_local_features = load_arff_features(
            get_data_and_meta_information_from_did(data_id),
            did=data_id,
        )

xml_local_features = parse_features_xml(
    features_to_xml(
        qualities_arff
    )
)

